# Text Embeddings and Similarity Measures

In Natural Language Processing (NLP), converting text into a numerical form is essential for downstream tasks such as classification, clustering, and search.

In this notebook, we will explore:

1. way: count the occurences of words. COUNT BASED APPROACHES. We count the frequency of words, that appear in documents, or a set of documents. Easy to compute, and easy to implement. 
DRAWBACKS: they do not capcture the semantic simillarity of words. For example car and automobile would be two differenct words, for BoW and TF-IDF. They ALSO can produce sparse vectors:
- Bag of Words (BoW)
- TF-IDF

---------------- separation of two approaches to text embeddings.

2. way: embedings. They include semantic understanding of the words, they are comparing. Car and automobile would be two same. They use embeddings, which produce much tighter vectors. BUT they are harder to integrate and they need much data to learn something. They capture deeper semantic structure:
- Word2Vec
- FastText

And these are the simmilarity measures
- Cosine Similarity
- Euclidean Distance

Each method has different strengths, and choosing the right one depends on the task and dataset. But first, let's review the basics.

## Basic definitions

The term vector originates from the Latin and has a meaning related to carry, bear, convey or transport. A vector is defined as a quantity that is characterized by both magnitude and direction, as opposed to a scalar, which is represented by a single numerical value. In an abstract sense, one can envision the vector as guiding them along the path indicated by the arrow that represents the vector. For example, vectors are used in Physics to represent a force by means of magnitude and direction.

Vectors are often visualized as arrows extending from an initial point, illustrating the displacement needed to transition from A to B. Let's suppose we have two points $A,B \in \mathbb{R}^2$, where $A(1,2)$ represents the initial point and $B(4,6)$ represents the terminal point of the vector as shown in the figure below. The length of the arrow corresponds to the vector's magnitude, while its orientation indicates the direction.

<img src="https://i.ibb.co/cX8ypVjw/untitled.jpg" alt="untitled" border="0">

A vector space is defined as a set of vectors that could be manipulated using algebraic operations from $\mathbb{R}^n$. Some examples of such operations are vector addition and scalar multiplication. All operations must adhere to the set of axioms, including associativity, commutativity, and distributivity.

The dimension of a vector refers to the number of independent components required to uniquely specify the vector within its vector space. High-dimensional vectors are representations that extend beyond three dimensions, making them challenging to visualize, as each dimension represents a unique feature or attribute. While low-dimensional vector representations (among other things) are suitable to model simpler problems, such as 2D or 3D space, high-dimensional vectors are crucial in several cutting-edge disciplines, including natural language processing, due to their granularity, which enables them to convey more information.

Embeddings are one of the pivotal concepts in natural language processing. They involve mapping words or documents to numerical representation in vector spaces. The process of generating embeddings typically employs well-known techniques like Word2Vec or BERT. These techniques leverage large corpora to capture semantic similarities by placing related words closer together in the vector space. The resulting high-dimensional vectors encapsulate contextual, syntactic, and semantic information, allowing algorithms to perform different tasks (e.g., text classification) more accurately.

## Bag of Words (BoW)

Bag of Words is one of the simplest text embedding methods. It represents each document as a vector based on the frequency of words, ignoring grammar and word order.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "The weather outside is nice today",
    "It is sunny today",
    "It is rainy",
    "It is raining today"     
]

# we choose that we want the vectorizer to counte the amount of times, the word has been found in the corpus.
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

bow_matrix = X.toarray()
print("Vocabulary:", vectorizer.vocabulary_)
print("BoW Matrix:\n", bow_matrix)


Vocabulary: {'the': 7, 'weather': 9, 'outside': 3, 'is': 0, 'nice': 2, 'today': 8, 'it': 1, 'sunny': 6, 'rainy': 5, 'raining': 4}
BoW Matrix:
 [[1 0 1 1 0 0 0 1 1 1]
 [1 1 0 0 0 0 1 0 1 0]
 [1 1 0 0 0 1 0 0 0 0]
 [1 1 0 0 1 0 0 0 1 0]]


PROBLEM S TEM APPROACHOM: word order is not captured! The semantics are not kept.

dog bites man\
man bites dog

these two sentences have way different meanings, but the bag of words, would produce the same output,
"dog" "bites" "man", which would get converted into bits. It just counts the words.

## Vector distances
With good text embeddings, vector distances can tell us, how similar two texts are. When desining embeddings, we try to make sure that text talking about simmilar things will be located close together.

**Euclidean Distance**

Euclidean distance measures the "straight-line" distance between two vectors in space. Unlike cosine similarity, it’s sensitive to vector magnitude. For example with bag-of-words vectors, the length of a sentence is important.

Evklidijsko razdaljo poračunamo samo kot razliko vseh bitov, aka [1,1,0] in [0,1,0] imata razdaljo 1, ker se samo po prvem bitu razlikujeta. [0,0,0] in [1,1,0] pa razdaljo 2.

$ Euccledean Distance = \sqrt{\sum_{i = 1}^{n} (v_1 - v_2)^2} $ 

**Cosine Similarity**

Cosine similarity measures the cosine of the angle between two vectors. It’s commonly used in NLP to compare text vectors.

Range: [-1, 1], where 1 means identical direction, and 0 means orthogonal.

The formula for **cosine similarity** between two vectors **A** and **B** is:

$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$

### Where:
- $ A \cdot B $ is the **dot product** of vectors A and B:
  $A \cdot B = \sum_{i=1}^{n} A_i \times B_i$
- $\|A\|$ and $\|B\|$ are the **Euclidean norms (magnitudes)** of A and B:
  $\|A\| = \sqrt{\sum_{i=1}^{n} A_i^2}, \quad \|B\| = \sqrt{\sum_{i=1}^{n} B_i^2}$

In [2]:
from scipy.spatial.distance import euclidean

vec1 = bow_matrix[0]
vec2 = bow_matrix[1]

print("Euclidean Distance (Bag of Words):", euclidean(vec1, vec2))


from sklearn.metrics.pairwise import cosine_similarity

vec1 = bow_matrix[0].reshape(1, -1)
vec2 = bow_matrix[1].reshape(1, -1)

print("Cosine Similarity (Bag of Words):", cosine_similarity(vec1, vec2)[0][0])

Euclidean Distance (Bag of Words): 2.449489742783178
Cosine Similarity (Bag of Words): 0.4082482904638631


## TF-IDF (Term Frequency - Inverse Document Frequency)

TF-IDF improves BoW by reducing the weight of common words and increasing the weight of rare but important words.\
It reduces the influence of common words, by multipling them with small weights (the IDF part).

TF (term frequency) = (number of times a word appears in a document) / (total words in document)
- Same as with bag of words, except normalized
- $TF(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$
- Example of TF:\
BoW: [1 0 1 1 0 0 0 1 1 1]\
TF:  [1/6 0 1/6 1/6 0 0 0 1/6 1/6 1/6]

IDF (inverse document frequency) = log(total number of documents / number of documents containing the word)
- $IDF(t) = \log \left( \frac{N}{n_t} \right)$

TF-IDF is calculated as a product of TF and IDF.

Sklearn library computes the TF-IDF value in a little bit different way, but with the same idea.
- TF = frequency of the word
- $IDF = log_n (\frac{N+1}{n_t + 1}) + 1$

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "The weather outside is nice today",
    "It is sunny today",
    "It is rainy",
    "It is raining today"     
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

print("TF-IDF Matrix:\n", X.toarray())
print("Feature Names:", vectorizer.get_feature_names_out())


TF-IDF Matrix:
 [[0.2412283  0.         0.46226355 0.46226355 0.         0.
  0.         0.46226355 0.29505684 0.46226355]
 [0.3612126  0.44181486 0.         0.         0.         0.
  0.69218835 0.         0.44181486 0.        ]
 [0.40264194 0.49248889 0.         0.         0.         0.77157901
  0.         0.         0.         0.        ]
 [0.3612126  0.44181486 0.         0.         0.69218835 0.
  0.         0.         0.44181486 0.        ]]
Feature Names: ['is' 'it' 'nice' 'outside' 'raining' 'rainy' 'sunny' 'the' 'today'
 'weather']


TF-IDF still has the same drawback as the BoW, of not including the order of the words, which IS important, as it gives much information about the words. So no semantics, the same as with the BoW approach.

## Word2Vec

Word2Vec learns word embeddings by training a neural network on a corpus to predict words based on their context (Skip-gram or CBOW).

Each word is mapped to a dense vector in a high-dimensional space, capturing semantic relationships.

<img src="https://i.ibb.co/6JPbNgdj/Word-Vectors.webp" alt="Word-Vectors" border="0">

While word2vec embeddings are designed as word embeddings, we can use them to represent an entire sentence by averaging the embeddings of all words in a sentence.

In [54]:
from gensim.models import Word2Vec
import numpy as np

# Tokenized corpus
tokenized_corpus = [
    sentence.split(" ") for sentence in corpus
]

# Train Word2Vec model (skip-gram)
w2v_model = Word2Vec(sentences=tokenized_corpus, vector_size=50, window=2, min_count=1, workers=2, sg=1)

# Example: Get vector for a single word
print("Vector for 'today':\n", w2v_model.wv['today'])

# --- Sentence Embedding by Averaging Word Vectors ---
def get_sentence_embedding(sentence, model):
    sentence_tokens = sentence.split(" ")
    word_vectors = [model.wv[word] for word in sentence_tokens if word in model.wv]
    if not word_vectors:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

# Example sentence
sentence = corpus[0]
sentence_embedding = get_sentence_embedding(sentence, w2v_model)

print("\nSentence embedding for '{}':\n{}".format(sentence, sentence_embedding))

Vector for 'today':
 [-0.01724032  0.00733188  0.01038034  0.01148451  0.01493466 -0.01233603
  0.00221135  0.01209523 -0.00568041 -0.01234772 -0.00082049 -0.01673882
 -0.01120064  0.01420986  0.00670545  0.01445213  0.01360124  0.01506231
 -0.00757872 -0.00112367  0.00469701 -0.00903856  0.01677838 -0.01971741
  0.01353002  0.00582915 -0.0098662   0.00879686 -0.00347934  0.01342351
  0.01993079 -0.00872537 -0.00119874 -0.0113919   0.00770207  0.00557356
  0.01378291  0.01220286  0.01907804  0.01854785  0.015797   -0.01397978
 -0.01831273 -0.00071154 -0.00620002  0.0157895   0.0118778  -0.0030915
  0.00302209  0.00358028]

Sentence embedding for 'The weather outside is nice today':
[-0.00541156  0.00628875  0.00109741 -0.00119988 -0.00035005 -0.00633097
  0.00126848  0.00813834 -0.00044633 -0.00700556  0.00505787 -0.00059451
 -0.0013911   0.00301572  0.00096892 -0.0025938   0.00818945  0.00868739
 -0.00707583 -0.00291372  0.00408076 -0.00271407  0.0093834   0.00768919
  0.00045347  0.0

In [56]:
sentence = "it is raining"
sentence_embedding = get_sentence_embedding(sentence, w2v_model)

# check, which sentence from corpus is the closest to the target sentence
for c in corpus:
    c_embedding = get_sentence_embedding(c, w2v_model)
    print(f"Cosine similarity between '{sentence}' and '{c}': {cosine_similarity([sentence_embedding], [c_embedding])[0][0]}")

Cosine similarity between 'it is raining' and 'The weather outside is nice today': 0.2810943126678467
Cosine similarity between 'it is raining' and 'It is sunny today': 0.2486116588115692
Cosine similarity between 'it is raining' and 'It is rainy': 0.26609453558921814
Cosine similarity between 'it is raining' and 'It is raining today': 0.6949722170829773


Bližje kot je 1.0, bližje sta si ta dva stavka.

## FastText

FastText, developed by Facebook, improves Word2Vec by learning representations for character n-grams. It handles out-of-vocabulary words better than Word2Vec.

n-grams are contiguous sequences of n characters extracted from a text. An example of n=3 for word "chat" would be: "cha" and "hat".

In [57]:
from gensim.models import FastText

ft_model = FastText(
    sentences=tokenized_corpus,
    vector_size=50,   # small but stable
    window=2,         # tight window to focus on local co-occurrence
    min_count=1,      # keep everything
    workers=1,
    sg=1,             # skip-gram
    min_n=3,          # subword range to help 'rainy' ~ 'raining'
    max_n=6,
    epochs=200        # more epochs for tiny data
)


print("Vector for 'raining':\n", ft_model.wv['raining'])
print("Vector for 'rainy' (OOV word):\n", ft_model.wv['rainy'])

Vector for 'raining':
 [-2.7772628e-03 -2.3407673e-03 -2.5894030e-03  2.4396596e-03
 -5.6364317e-03  2.5442985e-03 -1.2296955e-03  4.9060639e-03
 -1.0713181e-03 -3.5070956e-03 -1.8170479e-03 -2.3323619e-03
  8.2807819e-04  1.4721394e-03  2.2038606e-03 -4.6860827e-03
  5.3430940e-03  2.6558375e-03  2.6866950e-03 -2.1256255e-03
  2.7138088e-03  1.9124572e-03 -2.8266228e-04  5.5005187e-03
 -3.2896870e-03  1.7349162e-03 -7.1437142e-05 -1.0033271e-03
 -1.1887836e-03  9.9050289e-04 -1.7155631e-03  4.0465426e-03
  2.3143725e-03 -1.6192837e-04  1.2856883e-03  4.7712811e-04
 -1.1091569e-03  1.0856023e-03  2.6486132e-03 -1.1706769e-03
 -3.1944702e-03  2.2449935e-04  4.3048193e-03 -4.5224898e-03
  1.7033375e-03 -1.7170596e-03 -9.1850449e-04 -7.5960258e-04
 -2.1128024e-03 -1.7696065e-03]
Vector for 'rainy' (OOV word):
 [ 1.0277320e-03  8.8399759e-04 -5.3533231e-04  2.5561671e-03
  4.7821095e-03  5.5631665e-03  1.7415377e-03 -5.2616658e-04
  2.4205141e-03  1.8664250e-03 -2.8663226e-03 -3.7716876e-0

In [59]:
sentence = "it is raining"
sentence_embedding = get_sentence_embedding(sentence, ft_model)

# check, which sentence from corpus is the closest to the target sentence
for c in corpus:
    c_embedding = get_sentence_embedding(c, ft_model)
    print(f"Cosine similarity between '{sentence}' and '{c}': {cosine_similarity([sentence_embedding], [c_embedding])[0][0]}")

Cosine similarity between 'it is raining' and 'The weather outside is nice today': 0.3896881639957428
Cosine similarity between 'it is raining' and 'It is sunny today': 0.3048018515110016
Cosine similarity between 'it is raining' and 'It is rainy': 0.2795363664627075
Cosine similarity between 'it is raining' and 'It is raining today': 0.4533154368400574
